# 01 — Data Preparation & EDA
### Who Gets to Do Nothing? (Team #37)

**Goal:** Load raw ATUS dat files (2010–2024, no 2020), merge respondent demographics
with activity summary time codes, explore key patterns, and save a clean analysis-ready dataset.

**Output:** `cleaned_EDA_data.csv` — 145,248 rows, full US population sample


## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

---
## Step 1 — Load Raw Data

We merge two ATUS files per year on TUCASEID:
- **`atusresp`** — respondent demographics, employment, earnings
- **`atussum`** — per-person daily minutes in every activity code, plus age, sex, race, education

2020 is excluded as ATUS was not conducted that year due to COVID.

In [ ]:
years = [y for y in range(2010, 2025) if y != 2020]

respondent_dfs = {}
actsummary_dfs = {}
merged_dfs     = {}
BASE = os.path.abspath(os.path.join(os.getcwd(), "../../"))


for year in years:

    respondent_dfs[year] = pd.read_csv(os.path.join(BASE, f'ATUS Data/atusresp_{year}.dat'))
    actsummary_dfs[year] = pd.read_csv(os.path.join(BASE, f'ATUS Data/atussum_{year}.dat'))
    respondent_dfs[year].columns = respondent_dfs[year].columns.str.upper()
    actsummary_dfs[year].columns = actsummary_dfs[year].columns.str.upper()

    merged_dfs[year] = pd.merge(
        respondent_dfs[year][[
            'TUCASEID', 'TRNUMHOU',
            'TELFS', 'TRDPFTPT', 'TRERNWA',
            'TUDIARYDAY', 'TRHOLIDAY',
            'TRSPPRES', 'TESPEMPNOT',
            'TRCHILDNUM', 'TRYHHCHILD', 'TRHHCHILD',
            'TRMJOCGR', 'TEMJOT', 'TESCHENR',
            'TUFINLWGT'
        ]],
        actsummary_dfs[year][[
            'TUCASEID', 'TEAGE', 'TESEX',
            'PTDTRACE', 'PEEDUCA',
            'T120301','T120302','T120303','T120304','T120305','T120306',
            'T120307','T120308','T120309','T120310','T120311','T120312',
            'T120313','T120399'
        ]],
        on='TUCASEID', how='inner'
    )
    merged_dfs[year]['YEAR'] = year

final_df = pd.concat(merged_dfs.values(), ignore_index=True)
print(f'Rows: {len(final_df):,}  |  Columns: {final_df.shape[1]}')
final_df.head(3)

---
## Step 2 — Create Total Leisure

`TOTAL_LEISURE` is the sum of minutes across all 14 passive/home-based leisure activity
codes (T1203xx) for each respondent's single diary day. Each T-code column is
pre-aggregated by ATUS; we sum them. Values range from 0 to 1440 minutes.

In [ ]:
leisure_cols = [
    'T120301','T120302','T120303','T120304','T120305','T120306',
    'T120307','T120308','T120309','T120310','T120311','T120312',
    'T120313','T120399'
]
final_df['TOTAL_LEISURE'] = final_df[leisure_cols].sum(axis=1)

print(final_df['TOTAL_LEISURE'].describe().round(1))
print(f'\nZero leisure: {(final_df["TOTAL_LEISURE"]==0).mean()*100:.1f}% of respondents')

---
## EDA 1 — Distribution of Total Leisure Time

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(final_df['TOTAL_LEISURE'], bins=60, color='steelblue', edgecolor='white', alpha=0.85)
mean_val   = final_df['TOTAL_LEISURE'].mean()
median_val = final_df['TOTAL_LEISURE'].median()
ax.axvline(mean_val,   color='red',    linestyle='--', linewidth=1.5, label=f'Mean: {mean_val:.0f} min')
ax.axvline(median_val, color='orange', linestyle='--', linewidth=1.5, label=f'Median: {median_val:.0f} min')
zero_pct = (final_df['TOTAL_LEISURE'] == 0).mean() * 100
ax.set_title(f'Distribution of Total Daily Leisure Time\n({zero_pct:.1f}% of respondents report zero leisure)')
ax.set_xlabel('Total Leisure (minutes per day)')
ax.set_ylabel('Number of Respondents')
ax.legend()
plt.tight_layout()
plt.show()

---
## EDA 2 — Null Analysis

The leisure T-code columns have no missing values. Core demographic variables also have no nulls.

In [ ]:
check_cols = leisure_cols + ['TEAGE','TESEX','TELFS','PTDTRACE','PEEDUCA',
                              'TRCHILDNUM','TRSPPRES','TRERNWA']
nulls = final_df[check_cols].isnull().sum()
print('Null counts per variable:')
print(nulls[nulls > 0] if nulls.sum() > 0 else 'No nulls — clean dataset.')
print(f'\nNote: TRERNWA contains {(final_df["TRERNWA"]==-1).sum():,} values of -1')
print('These are structural blanks (not applicable) per the ATUS data dictionary,')
print('not missing responses. They are explored more in detail in Notebook 2.')

---
## EDA 3 — Correlation Matrix

Pearson correlations across key numeric variables before any encoding.


In [ ]:
corr_cols = {
    'TOTAL_LEISURE': 'Leisure Time',
    'YEAR':          'Year',
    'TEAGE':         'Age',
    'PEEDUCA':       'Education',
    'TRCHILDNUM':    'Num Children',
    'TRERNWA':       'Weekly Earnings',
    'TESEX':         'Sex',
    'TELFS':         'Employment',
    'PTDTRACE':      'Race',
    'TRSPPRES':      'Partner Status',
}
corr = final_df[list(corr_cols.keys())].rename(columns=corr_cols).corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, vmin=-1, vmax=1, annot_kws={'size': 9}, ax=ax)
ax.set_title('Correlation Matrix — ATUS Dataset (raw values)', fontsize=13, pad=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

---
## EDA 4 — Leisure by Day of Week

Each respondent was surveyed about a single day.

In [ ]:
day_map   = {1:'Sun', 2:'Mon', 3:'Tue', 4:'Wed', 5:'Thu', 6:'Fri', 7:'Sat'}
day_order = ['Sun','Mon','Tue','Wed','Thu','Fri','Sat']
day_colors = ['#e8622a' if d in ['Sun','Sat'] else 'steelblue' for d in day_order]
day_means = (final_df.groupby(final_df['TUDIARYDAY'].map(day_map))['TOTAL_LEISURE']
             .mean().reindex(day_order))
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(day_means.index, day_means.values, color=day_colors, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, day_means.values):
    ax.text(bar.get_x()+bar.get_width()/2, val+2, f'{val:.0f}',
            ha='center', va='bottom', fontsize=9)
ax.set_title('Average Daily Leisure by Day of Week\n'
             'Weekend respondents report ~60 more minutes than weekday respondents')
ax.set_xlabel('Day of Week')
ax.set_ylabel('Average Leisure (minutes)')
plt.tight_layout()
plt.show()

---
## EDA 5 — Leisure by Employment Status

In [ ]:
emp_map = {1:'Employed (at work)', 2:'Employed (absent)',
           3:'Unemployed (layoff)', 4:'Unemployed (looking)', 5:'Not in labor force'}
emp_means = (final_df.groupby(final_df['TELFS'].map(emp_map))['TOTAL_LEISURE']
             .mean().sort_values())
colors = sns.color_palette('RdYlGn', len(emp_means))
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(emp_means.index, emp_means.values, color=colors, edgecolor='white', alpha=0.9)
for bar, val in zip(bars, emp_means.values):
    ax.text(val-8, bar.get_y()+bar.get_height()/2, f'{val:.0f} min',
            va='center', ha='right', color='white', fontweight='bold', fontsize=9)
ax.set_title('Average Leisure by Employment Status', fontsize=13)
ax.set_xlabel('Average Leisure (minutes per day)')
plt.tight_layout()
plt.show()

---
## EDA 6 — Leisure by Sex × Age Group

In [ ]:
age_order  = ['<18','18-24','25-34','35-44','45-54','55-64','65+']
sex_map    = {1:'Male', 2:'Female'}
age_groups = pd.cut(final_df['TEAGE'], bins=[0,17,24,34,44,54,64,120],
                    labels=age_order, right=True)
pivot = (final_df.groupby([final_df['TESEX'].map(sex_map), age_groups],
                           observed=False)['TOTAL_LEISURE']
         .mean().unstack().reindex(columns=age_order))
diff  = pivot.loc['Male'] - pivot.loc['Female']
fig, axes = plt.subplots(2, 1, figsize=(11, 7), gridspec_kw={'height_ratios':[3,1]})
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd_r', linewidths=0.5, ax=axes[0],
            cbar_kws={'label':'Mean Leisure (min)'})
axes[0].set_title('Average Daily Leisure (min) by Sex and Age Group')
axes[0].set_xlabel('')
axes[0].set_ylabel('')
gap_df = diff.to_frame().T
gap_df.index = ['Male vs Female gap']
sns.heatmap(gap_df, annot=True, fmt='.0f', cmap='RdBu', center=0,
            linewidths=0.5, ax=axes[1], cbar_kws={'label':'Minutes'})
axes[1].set_title('Gender Gap in Leisure (positive = males have more)')
axes[1].set_xlabel('Age Group')
plt.tight_layout()
plt.show()

---
## EDA 7 — Age Curve

Plotting average leisure at each individual age confirms the U-shape: leisure is
high in youth, dips around age 35 as career and family demands peak, then rises
steeply into retirement. A linear term cannot fit this, it justifies `TEAGE²`.

In [ ]:
age_means = final_df.groupby('TEAGE')['TOTAL_LEISURE'].mean().reset_index()
fig, ax   = plt.subplots(figsize=(10, 5))
ax.plot(age_means['TEAGE'], age_means['TOTAL_LEISURE'], color='steelblue', linewidth=2)
ax.axvline(x=35, color='#D85A30', linestyle='--', linewidth=1.5, label='~Age 35 (leisure minimum)')
ax.set_title('Average Leisure by Age — U-shaped relationship, not linear')
ax.set_xlabel('Age')
ax.set_ylabel('Average Leisure (minutes)')
ax.legend()
plt.tight_layout()
plt.show()

---
## EDA 8 — Leisure by Race

Raw average leisure differs across racial groups. After controlling for employment,
income, children and other factors in the regression, any remaining race coefficient
represents a structural residual disparity.

We consolidate `PTDTRACE` into 4 groups to avoid sparse dummies:
White (1), Black (2), Asian (4), Other/Multiracial (Other).

In [ ]:
def consolidate_race(code):
    if code == 1: return 'White'
    elif code == 2: return 'Black'
    elif code == 4: return 'Asian'
    else: return 'Other/Multiracial'

final_df['RACE_LABEL'] = final_df['PTDTRACE'].apply(consolidate_race)
race_order = ['White','Black','Asian','Other/Multiracial']
race_means = final_df.groupby('RACE_LABEL')['TOTAL_LEISURE'].mean().reindex(race_order)
race_counts = final_df['RACE_LABEL'].value_counts().reindex(race_order)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors_race = ['#5DCAA5','#1D9E75','#D85A30','#F0997B']
bars = axes[0].bar(race_order, race_means.values, color=colors_race, edgecolor='white', alpha=0.9)
for bar, val in zip(bars, race_means.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, val+2, f'{val:.0f}',
                 ha='center', va='bottom', fontsize=9)
axes[0].set_title('Average Leisure by Race (raw, unadjusted)')
axes[0].set_ylabel('Average Leisure (minutes)')
axes[1].bar(race_order, race_counts.values, color=colors_race, edgecolor='white', alpha=0.9)
for bar, val in zip(axes[1].patches, race_counts.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, val+200, f'{val:,}',
                 ha='center', va='bottom', fontsize=9)
axes[1].set_title('Sample size by race group')
axes[1].set_ylabel('Respondents')
plt.suptitle('Race differences in leisure — regression controls for confounders', fontsize=10, y=1.01)
plt.tight_layout()
plt.show()

---
## EDA 9 — Leisure by Education

We consolidate `PEEDUCA` into 5 groups: Less than HS (<39), HS/GED (39),
Some college (40–42), Bachelor's (43), Graduate (44–46).

In [ ]:
def consolidate_educ(code):
    if code < 39:          return '1_LessThanHS'
    elif code == 39:       return '2_HS/GED'
    elif 40 <= code <= 42: return '3_SomeCollege'
    elif code == 43:       return '4_Bachelors'
    else:                  return '5_Graduate'

final_df['EDUC_LABEL'] = final_df['PEEDUCA'].apply(consolidate_educ)
educ_order  = ['1_LessThanHS','2_HS/GED','3_SomeCollege','4_Bachelors','5_Graduate']
educ_labels = ['< HS','HS/GED','Some college',"Bachelor's",'Graduate']
educ_means  = final_df.groupby('EDUC_LABEL')['TOTAL_LEISURE'].mean().reindex(educ_order)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors_educ = sns.color_palette('Blues_r', len(educ_order))
bars = axes[0].bar(educ_labels, educ_means.values, color=colors_educ, edgecolor='white', alpha=0.9)
for bar, val in zip(bars, educ_means.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, val+2, f'{val:.0f}',
                 ha='center', va='bottom', fontsize=9)
axes[0].set_title('Average Leisure by Education (raw)\nNegative gradient — more education = less leisure')
axes[0].set_ylabel('Average Leisure (minutes)')
axes[1].plot(range(len(educ_order)), educ_means.values, marker='o', markersize=8,
             color='steelblue', linewidth=2)
axes[1].set_xticks(range(len(educ_order)))
axes[1].set_xticklabels(educ_labels)
for i, val in enumerate(educ_means.values):
    axes[1].text(i, val+2, f'{val:.0f}', ha='center', va='bottom', fontsize=9)
axes[1].set_title('Education gradient — consistent downward trend')
axes[1].set_ylabel('Average Leisure (minutes)')
plt.tight_layout()
plt.show()

---
## EDA 10 — Leisure by Number of Children
This justifies using the continuous count
rather than a binary 'has children' flag.

In [ ]:
child_means = (final_df[final_df['TRCHILDNUM']<=5]
               .groupby('TRCHILDNUM')['TOTAL_LEISURE'].mean())
child_counts = (final_df[final_df['TRCHILDNUM']<=5]
                .groupby('TRCHILDNUM')['TOTAL_LEISURE'].count())
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(child_means.index.astype(str), child_means.values,
              color='steelblue', edgecolor='white', alpha=0.85)
for bar, val, cnt in zip(bars, child_means.values, child_counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, val+2,
            f'{val:.0f} min\n(n={cnt:,})', ha='center', va='bottom', fontsize=8)
ax.set_title('Average Leisure by Number of Household Children\nEach child costs ~15-20 minutes')
ax.set_xlabel('Number of children under 18')
ax.set_ylabel('Average Leisure (minutes)')
plt.tight_layout()
plt.show()

---
## EDA 11 — Trends Over Time

Average leisure by year shows a clear structural shift around 2021, likely driven
by remote work adoption and post-COVID labour market changes. This motivates
including `YEAR` dummies rather than treating time as a linear trend.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
avg_by_year = final_df.groupby('YEAR')['TOTAL_LEISURE'].mean().reset_index()
axes[0].plot(avg_by_year['YEAR'], avg_by_year['TOTAL_LEISURE'],
             marker='o', markersize=4, color='steelblue', linewidth=2)
axes[0].axvline(x=2021, color='#D85A30', linestyle='--', linewidth=1.5, label='2021 shift')
axes[0].set_title('Average Daily Leisure by Year\nStructural shift post-2021')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Average Leisure (minutes)')
axes[0].legend()
sex_map = {1:'Male', 2:'Female'}
trend_sex = (final_df.groupby(['YEAR', final_df['TESEX'].map(sex_map)])['TOTAL_LEISURE']
             .mean().reset_index())
trend_sex.columns = ['YEAR','SEX','mean']
for grp, color in [('Male','steelblue'),('Female','#e8622a')]:
    sub = trend_sex[trend_sex['SEX']==grp]
    axes[1].plot(sub['YEAR'], sub['mean'], marker='o', markersize=3,
                 label=grp, color=color, linewidth=2)
axes[1].axvline(x=2021, color='grey', linestyle='--', linewidth=1, alpha=0.6)
axes[1].set_title('Trend by Sex — gender gap persists across all years')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Average Leisure (minutes)')
axes[1].legend()
plt.tight_layout()
plt.show()


Save the full dataset for Notebook 2. This avoids rerunning the data loading loop.

In [ ]:
final_df.to_csv('cleaned_EDA_data.csv', index=False)
print(f'Saved: cleaned_EDA_data.csv  ({len(final_df):,} rows, {final_df.shape[1]} columns)')